# Simulated Annealing (SA) for Minimization

**Problem:** Minimize $f(x) = x^2 - 6x + 13$ using the Simulated Annealing algorithm.

**Given settings**

| Parameter | Value |
|---|---|
| Initial solution | $x = 1$ |
| Initial temperature | $T = 10$ |
| Cooling factor | $\alpha = 0.5$ |
| Minimum temperature | $T_{min} = 1$ |
| Neighbour rule | $x' = x \pm 1$ |
| Random numbers (for accepting worse moves) | $r_1=0.30,\; r_2=0.80,\; r_3=0.20,\; r_4=0.90$ |

Acceptance rule:
- If $\Delta f = f(x') - f(x) < 0$ (neighbour is better) → **always accept**.
- If $\Delta f \ge 0$ (neighbour is worse) → compute $P = e^{-\Delta f / T}$ and accept **only if** the next random number $r < P$; otherwise reject and stay at the current $x$.


## Algorithm (for the lab record)

```
ALGORITHM: Simulated Annealing (Minimization)

1.  Define the objective function f(x).
2.  Initialize:
        x        <- initial solution
        T        <- initial temperature
        alpha    <- cooling factor (0 < alpha < 1)
        T_min    <- stopping temperature
        best_x   <- x
        best_f   <- f(x)

3.  WHILE T >= T_min DO
        3.1  Generate a neighbour solution  x' = x ± 1
        3.2  Compute  delta_f = f(x') - f(x)

        3.3  IF delta_f < 0 THEN
                 Accept x'  (it is an improvement)
             ELSE
                 Compute acceptance probability  P = exp(-delta_f / T)
                 Draw / take next random number r  in [0, 1]
                 IF r < P THEN
                      Accept x'  (accept a worse solution, to escape local optima)
                 ELSE
                      Reject x'  (keep current x)

        3.4  IF f(x) < best_f THEN
                 best_x <- x ; best_f <- f(x)   (update best-so-far)

        3.5  Cool down the temperature:  T <- alpha * T

4.  RETURN best_x, best_f
```


## Step 0 — Objective function

Implement $f(x) = x^2 - 6x + 13$ and inspect it before running SA.

In [1]:
def f(x):
    """Objective function to be minimized."""
    return x**2 - 6*x + 13

# Quick sanity check
print("f(1) =", f(1))
print("f(2) =", f(2))
print("f(3) =", f(3))


f(1) = 8
f(2) = 5
f(3) = 4


## Step 1 — Evaluate the initial solution

$x = 1 \Rightarrow f(x) = 1^2 - 6(1) + 13 = 8$

In [2]:
x0 = 1
print(f"f({x0}) = {f(x0)}")


f(1) = 8


## Step 2 & 3 — First neighbour and acceptance check

Neighbour: $x' = 2 \Rightarrow f(x') = 4 - 12 + 13 = 5$

$\Delta f = f(x') - f(x) = 5 - 8 = -3$

Since $\Delta f < 0$, the neighbour is **better**, so it is accepted unconditionally
(no random number is needed for an improving move).

In [3]:
x_prime = 2
delta_f = f(x_prime) - f(x0)
print(f"x' = {x_prime}, f(x') = {f(x_prime)}")
print(f"Delta f = {delta_f}")
print("Decision:", "ACCEPT (improvement)" if delta_f < 0 else "Needs probability test")


x' = 2, f(x') = 5
Delta f = -3
Decision: ACCEPT (improvement)


## Step 4 & 5 — Full Simulated Annealing run (4 iterations) + iteration table

Neighbour generation is continued the same way it started, i.e. $x' = x + 1$ each
iteration (moving one step at a time), and the temperature is cooled geometrically
as $T \leftarrow \alpha T$ after every iteration.

The loop stops as soon as $T < T_{min}$. Starting at $T=10$ with $\alpha = 0.5$:

$$10 \to 5 \to 2.5 \to 1.25 \to 0.625\;(<T_{min}=1,\ \text{stop})$$

so the run naturally lasts **exactly four iterations** — matching the problem statement.

The provided random numbers $r_1, r_2, r_3, r_4$ are consumed **in order, only when a
worse neighbour is proposed** (i.e. only when $\Delta f \ge 0$).

In [4]:
import math
import pandas as pd

# ---- SA parameters ----
x = 1                 # initial solution
T = 10                # initial temperature
alpha = 0.5           # cooling factor
T_min = 1             # minimum temperature
random_numbers = [0.30, 0.80, 0.20, 0.90]   # r1, r2, r3, r4
r_index = 0                                  # pointer into random_numbers

best_x, best_f = x, f(x)
records = []
iteration = 0

while T >= T_min:
    iteration += 1

    x_current  = x
    x_neighbor = x_current + 1          # neighbour rule x' = x + 1
    fx         = f(x_current)
    fx_new     = f(x_neighbor)
    delta_f    = fx_new - fx

    if delta_f < 0:
        # Improving move -> always accept, no random number consumed
        P = 1.0
        r_used = None
        decision = "Accept"
        x = x_neighbor
    else:
        # Worse move -> probabilistic acceptance
        P = math.exp(-delta_f / T)
        r_used = random_numbers[r_index]
        r_index += 1
        if r_used < P:
            decision = "Accept"
            x = x_neighbor
        else:
            decision = "Reject"
            # x stays at x_current

    if f(x) < best_f:
        best_x, best_f = x, f(x)

    records.append({
        "Iteration": iteration,
        "T": T,
        "Current x": x_current,
        "Neighbour x'": x_neighbor,
        "f(x)": fx,
        "f(x')": fx_new,
        "Delta f": delta_f,
        "P = exp(-Δf/T)": round(P, 4),
        "Random r": r_used,
        "Decision": decision,
        "Best x": best_x,
        "Best f(x)": best_f,
    })

    T = T * alpha   # cool down

df = pd.DataFrame(records)
df


,Iteration,T,Current x,Neighbour x',f(x),f(x'),Delta f,P = exp(-Δf/T),Random r,Decision,Best x,Best f(x)
0,1,10.00,1,2,8,5,-3,1.0000,NaN,Accept,2,5
1,2,5.00,2,3,5,4,-1,1.0000,NaN,Accept,3,4
2,3,2.50,3,4,4,5,1,0.6703,0.3,Accept,3,4
3,4,1.25,4,5,5,8,3,0.0907,0.8,Reject,3,4


### Formatted iteration table

In [5]:
pd.set_option('display.max_columns', None)
df.style.set_caption("Simulated Annealing – Iteration Table")


,Iteration,T,Current x,Neighbour x',f(x),f(x'),Delta f,P = exp(-Δf/T),Random r,Decision,Best x,Best f(x)
0,1,10.000000,1,2,8,5,-3,1.000000,nan,Accept,2,5
1,2,5.000000,2,3,5,4,-1,1.000000,nan,Accept,3,4
2,3,2.500000,3,4,4,5,1,0.670300,0.300000,Accept,3,4
3,4,1.250000,4,5,5,8,3,0.090700,0.800000,Reject,3,4


## Step 6 — Best solution found by SA

The best (lowest-cost) solution encountered during the run is printed below.

In [6]:
print(f"Best x found by SA = {best_x}")
print(f"Best f(x) found by SA = {best_f}")


Best x found by SA = 3
Best f(x) found by SA = 4


## Step 7 — Analytical verification

$$f(x) = x^2 - 6x + 13$$
$$f'(x) = 2x - 6 = 0 \implies x = 3$$
$$f''(x) = 2 > 0 \implies x = 3 \text{ is a minimum}$$
$$f(3) = 9 - 18 + 13 = 4$$

So the true analytical minimum is $x^\* = 3,\; f(x^\*) = 4$, which **matches** the
best solution found by the Simulated Annealing run above.

In [7]:
import sympy as sp

x_sym = sp.symbols('x')
f_sym = x_sym**2 - 6*x_sym + 13

f_prime = sp.diff(f_sym, x_sym)
critical_points = sp.solve(sp.Eq(f_prime, 0), x_sym)
f_double_prime = sp.diff(f_sym, x_sym, 2)

print("f'(x)  =", f_prime)
print("Critical point(s):", critical_points)
print("f''(x) =", f_double_prime, "  (positive => minimum)")
for cp in critical_points:
    print(f"f({cp}) = {f_sym.subs(x_sym, cp)}")


f'(x)  = 2*x - 6
Critical point(s): [3]
f''(x) = 2   (positive => minimum)
f(3) = 4


## Interpretation of Results

- Starting from $x = 1$ (where $f(x)=8$), the algorithm moved **downhill** for the
  first two iterations ($x=1\to2\to3$), since each neighbour genuinely reduced
  the objective value — these moves needed no randomness at all.
- At iteration 3, the current solution had already reached the true optimum
  $x=3$. The next neighbour $x'=4$ was **worse** ($\Delta f = 1 > 0$). SA computed
  $P=e^{-1/2.5}\approx 0.6703$ and compared it with $r_1=0.30$. Since
  $r_1 < P$, the worse move was **accepted** — this is the defining feature of
  Simulated Annealing: at high-ish temperatures it deliberately accepts some
  uphill moves so that it does not get permanently trapped in whatever local
  optimum it currently sits at.
- At iteration 4, the temperature had cooled to $T=1.25$, and a further worse
  move to $x'=5$ ($\Delta f = 3$) gave a much smaller acceptance probability
  $P=e^{-3/1.25}\approx 0.0907$. This time $r_2 = 0.80 > P$, so the move was
  **rejected** and the search stayed at $x=4$. This shows the expected SA
  behaviour: as $T$ decreases, the algorithm becomes progressively less willing
  to accept worse solutions and behaves more like plain hill-climbing.
- The temperature fell below $T_{min}=1$ right after iteration 4
  ($T = 1.25 \times 0.5 = 0.625$), so the algorithm stopped there. Random
  numbers $r_3$ and $r_4$ were not needed because only two worse neighbours
  ($x'=4$ and $x'=5$) were ever proposed in this short run — if the run had
  continued longer (i.e. with a slower cooling schedule / lower $T_{min}$),
  $r_3$ and $r_4$ would have been used for the next worse-move decisions in
  exactly the same way.
- Even though the search *wandered* off to $x=4$ by the end (because it
  accepted one worse move and wasn't offered a chance to correct it before
  stopping), SA keeps track of the **best-so-far** solution separately from
  the current solution. That best-so-far value is $x=3,\ f(x)=4$.
- This matches exactly the analytical minimum obtained by setting
  $f'(x)=2x-6=0 \Rightarrow x=3$, confirming that Simulated Annealing
  correctly located the global minimum of this convex function, and also
  illustrating why it is useful for harder, non-convex problems: it is
  willing to temporarily accept worse solutions in order to avoid getting
  stuck, while still remembering the best solution it has seen.
